# Runtime Safety Filter (CBF)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/PythonInteractiveRobotics/blob/main/notebooks/safety_filter_cbf.ipynb)

A naive go-to-goal controller does not know about obstacles. A separate safety filter projects each velocity command back into a CBF-style safe set.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path("/content/PythonInteractiveRobotics")
    if not ROOT.exists():
        subprocess.run([
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/rsasaki0109/PythonInteractiveRobotics.git",
            str(ROOT),
        ], check=True)

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "."], check=True)
print(f"ready: {ROOT}")

In [ ]:
import importlib.util
import sys
from pathlib import Path

def load_example(relative_path: str):
    path = Path(relative_path)
    spec = importlib.util.spec_from_file_location(path.stem, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

module = load_example("examples/navigation/29_safety_filter_cbf.py")
trace = module.run(seed=0, render=False, max_steps=120)
summary = trace.summary()
print(summary)
print("final closest approach:", summary.final_info["closest_approach"])
print("filter activations:", summary.final_info["filter_active_count"])

In [ ]:
from IPython.display import Image, display

display(Image(filename="docs/assets/gifs/safety_filter_cbf.gif"))

In [ ]:
# Inspect the first steps where the filter changed the nominal command.
for step, info in enumerate(trace.infos):
    if info.get("filter_active"):
        print(
            f"step={step} h_min={info['barrier_h_min']:.3f} "
            f"u_nominal={info['u_nominal']} u_safe={info['u_safe']}"
        )
        if step > 12:
            break